In [41]:
import os
import shutil
import zipfile
import rasterio
import datetime
import numpy as np
from pathlib import Path
from shapely.geometry import box, Polygon
from eodag import EODataAccessGateway, setup_logging

In [15]:
# LOGGING (DEBUG)
setup_logging(verbose=2)

# AUTHENTIFICATION
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__USERNAME"] = "damba.kone@umontpellier.fr"
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__PASSWORD"] = "0767991488Dk@"
os.environ["EODAG__COP_DATASPACE__PRIORITY"] = "1"

In [16]:
dag = EODataAccessGateway()
dag.set_preferred_provider("cop_dataspace")

# DOSSIER DE SORTIE
output_dir = Path("downloads") / "Sentinel-2_Cayenne_2023"
output_dir.mkdir(parents=True, exist_ok=True)


# ZONE D’INTÉRÊT (Cayenne)
roi = box(-53.5, 4.5, -51.5, 6.5)


# RECHERCHE
print(" Recherche des images Sentinel-2 L2A...")

search_results = dag.search(
    collection="SENTINEL-2", 
    productType="S2MSI2A",   
    geom=roi,
    start="2023-01-01",
    end="2023-12-31",
    limit=5,
    provider="cop_dataspace"   
)

print(f" {len(search_results)} images trouvées")

2026-04-08 10:49:02,068 eodag.provider                   [INFO    ] Loading user configuration from: C:\Users\Kone\.config\eodag\eodag.yml
2026-04-08 10:49:02,073 eodag.provider                   [WARNING ] providers: skipped creating due to invalid config
2026-04-08 10:49:02,101 eodag.core                       [INFO    ] usgs: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 10:49:02,104 eodag.core                       [INFO    ] aws_eos: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 10:49:02,105 eodag.core                       [INFO    ] cop_ads: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 10:49:02,105 eodag.core                       [INFO    ] cop_cds: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 10:49:02,106 eodag.core                       [INFO    ] meteoblue: provider ne

 Recherche des images Sentinel-2 L2A...
 5 images trouvées


In [17]:
print("\n--- Aperçu des images ---")

for p in search_results:
    print("Titre :", p.properties.get("title"))
    print("Date :", p.properties.get("start_datetime"))
    print("Cloud cover :", p.properties.get("cloudCover"))
    print("-" * 40)


--- Aperçu des images ---
Titre : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809
Date : 2023-01-01T13:56:59.024000Z
Cloud cover : None
----------------------------------------
Titre : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCM_20240809T223809
Date : 2023-01-01T13:56:59.024000Z
Cloud cover : None
----------------------------------------
Titre : S2B_MSIL2A_20230101T135659_N0510_R067_T22NDL_20240809T223809
Date : 2023-01-01T13:56:59.024000Z
Cloud cover : None
----------------------------------------
Titre : S2B_MSIL2A_20230101T135659_N0510_R067_T22NBK_20240809T223809
Date : 2023-01-01T13:56:59.024000Z
Cloud cover : None
----------------------------------------
Titre : S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809
Date : 2023-01-01T13:56:59.024000Z
Cloud cover : None
----------------------------------------


In [18]:
# DOWNLOAD 
print("\n Téléchargement des images...")

for i, product in enumerate(search_results):
    try:
        print(f"\n[{i+1}/{len(search_results)}] Téléchargement : {product.properties.get('title')}")
        dag.download(
            product,
            outputs_prefix=str(output_dir),
            extract=False  
        )
        print(" Succès")

    except Exception as e:
        print(f" Erreur : {e}")

print("\n Téléchargement terminé !")


 Téléchargement des images...

[1/5] Téléchargement : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809


0.00B [00:00, ?B/s]

2026-04-08 10:49:16,904 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(34010a71-575c-45cf-95cb-094d9839cfd4)/$value
2026-04-08 10:49:16,909 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809.zip
2026-04-08 10:49:16,912 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 10:49:16,913 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(34010a71-575c-45cf-95cb-094d9839cfd4)/$value


 Succès

[2/5] Téléchargement : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCM_20240809T223809


0.00B [00:00, ?B/s]

...

2026-04-08 10:49:17,011 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(62b3e921-38b3-4161-927f-f49f93275ece)/$value
2026-04-08 10:49:17,015 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBK_20240809T223809.zip
2026-04-08 10:49:17,017 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 10:49:17,025 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(62b3e921-38b3-4161-927f-f49f93275ece)/$value


 Succès

[5/5] Téléchargement : S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809


0.00B [00:00, ?B/s]

2026-04-08 10:49:17,055 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(7a32ece2-68ac-4b0b-b698-9f065ca28a2b)/$value
2026-04-08 10:49:17,063 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809.zip
2026-04-08 10:49:17,068 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 10:49:17,075 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(7a32ece2-68ac-4b0b-b698-9f065ca28a2b)/$value


 Succès

 Téléchargement terminé !


In [ ]:
print("\n Téléchargement des images...")

for i, product in enumerate(search_results):
    try:
        print(f"\n[{i+1}/{len(search_results)}] Téléchargement : {product.properties.get('title')}")
        dag.download(product, outputs_prefix=str(output_dir))
        print(" Succès !")
    except Exception as e:
        print(f" Erreur : {e}")

print("\n Téléchargement terminé !")

In [19]:
# LOGGING
setup_logging(verbose=2)


# AUTHENTIFICATION
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__USERNAME"] = "damba.kone@umontpellier.fr"
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__PASSWORD"] = "0767991488Dk@"


# INIT EODAG
dag = EODataAccessGateway()
dag.set_preferred_provider("cop_dataspace")


# DOSSIERS
output_dir = Path("data/raw/Sentinel2")
zip_dir = output_dir / "zip"
extract_dir = output_dir / "SAFE"

zip_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)


# ROI (Cayenne)
roi = box(-53.5, 4.5, -51.5, 6.5)

print("Recherche des images...")

search_results = dag.search(
    collection="SENTINEL-2",
    productType="S2MSI2A",
    geom=roi,
    start="2023-01-01",
    end="2023-12-31",
    limit=5,
    provider="cop_dataspace"
)

print(f"{len(search_results)} images trouvées")

2026-04-08 11:01:09,611 eodag.provider                   [INFO    ] Loading user configuration from: C:\Users\Kone\.config\eodag\eodag.yml
2026-04-08 11:01:09,616 eodag.provider                   [WARNING ] providers: skipped creating due to invalid config
2026-04-08 11:01:09,648 eodag.core                       [INFO    ] usgs: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 11:01:09,651 eodag.core                       [INFO    ] aws_eos: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 11:01:09,653 eodag.core                       [INFO    ] cop_ads: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 11:01:09,654 eodag.core                       [INFO    ] cop_cds: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 11:01:09,656 eodag.core                       [INFO    ] meteoblue: provider ne

Recherche des images...
5 images trouvées


In [20]:
# DOWNLOAD 
print("\nTéléchargement des ZIP...")

downloaded_files = []

for i, product in enumerate(search_results):
    try:
        print(f"[{i+1}/{len(search_results)}] {product.properties.get('title')}")

        path = dag.download(
            product,
            outputs_prefix=str(zip_dir),
            extract=False  
        )

        downloaded_files.append(path)

        print("OK téléchargé :", path)

    except Exception as e:
        print("Erreur téléchargement :", e)


Téléchargement des ZIP...
[1/5] S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809


0.00B [00:00, ?B/s]

2026-04-08 11:01:59,071 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(34010a71-575c-45cf-95cb-094d9839cfd4)/$value
2026-04-08 11:01:59,074 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809.zip
2026-04-08 11:01:59,078 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 11:01:59,079 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(34010a71-575c-45cf-95cb-094d9839cfd4)/$value


OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809.zip
[2/5] S2B_MSIL2A_20230101T135659_N0510_R067_T22NCM_20240809T223809


0.00B [00:00, ?B/s]

...

2026-04-08 11:01:59,132 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(62b3e921-38b3-4161-927f-f49f93275ece)/$value
2026-04-08 11:01:59,138 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBK_20240809T223809.zip
2026-04-08 11:01:59,139 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 11:01:59,140 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(62b3e921-38b3-4161-927f-f49f93275ece)/$value


OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBK_20240809T223809.zip
[5/5] S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809


0.00B [00:00, ?B/s]

2026-04-08 11:01:59,149 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(7a32ece2-68ac-4b0b-b698-9f065ca28a2b)/$value
2026-04-08 11:01:59,156 eodag.download.base              [INFO    ] Product already downloaded: C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809.zip
2026-04-08 11:01:59,158 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 11:01:59,160 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(7a32ece2-68ac-4b0b-b698-9f065ca28a2b)/$value


OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809.zip


In [21]:
# EXTRACTION MANUELLE
print("\nExtraction des ZIP...")

for zip_path in downloaded_files:
    try:
        zip_path = Path(zip_path)

        print(f"Extraction : {zip_path.name}")

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print("OK extrait")

    except Exception as e:
        print("Erreur extraction :", e)

print("\nPipeline terminé proprement !")


Extraction des ZIP...
Extraction : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCK_20240809T223809.zip
OK extrait
Extraction : S2B_MSIL2A_20230101T135659_N0510_R067_T22NCM_20240809T223809.zip
OK extrait
Extraction : S2B_MSIL2A_20230101T135659_N0510_R067_T22NDL_20240809T223809.zip
OK extrait
Extraction : S2B_MSIL2A_20230101T135659_N0510_R067_T22NBK_20240809T223809.zip
OK extrait
Extraction : S2B_MSIL2A_20230101T135659_N0510_R067_T22NBM_20240809T223809.zip
OK extrait

Pipeline terminé proprement !


In [43]:
# LOGGING
setup_logging(verbose=2)


# AUTHENTIFICATION
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__USERNAME"] = "damba.kone@umontpellier.fr"
os.environ["EODAG__COP_DATASPACE__AUTH__CREDENTIALS__PASSWORD"] = "0767991488Dk@"


# INIT EODAG
dag = EODataAccessGateway()
dag.set_preferred_provider("cop_dataspace")


# DOSSIERS
output_dir = Path("data/raw_filtered/Sentinel2")
zip_dir = output_dir / "zip"
extract_dir = output_dir / "SAFE"

zip_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)


# ROI (Cayenne)
roi = box(-53.5, 4.5, -51.5, 6.5)

print("Recherche des images...")

search_results = dag.search(
    collection="SENTINEL-2",
    productType="S2MSI2A",
    geom=roi,
    start="2023-06-01",
    end="2023-12-31",
    limit=20,
    provider="cop_dataspace"
)

print(f"{len(search_results)} images trouvées")

2026-04-08 13:47:37,239 eodag.provider                   [INFO    ] Loading user configuration from: C:\Users\Kone\.config\eodag\eodag.yml
2026-04-08 13:47:37,245 eodag.provider                   [WARNING ] providers: skipped creating due to invalid config
2026-04-08 13:47:37,269 eodag.core                       [INFO    ] usgs: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 13:47:37,272 eodag.core                       [INFO    ] aws_eos: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 13:47:37,274 eodag.core                       [INFO    ] cop_ads: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 13:47:37,275 eodag.core                       [INFO    ] cop_cds: provider needing auth for search has been pruned because no credentials could be found
2026-04-08 13:47:37,276 eodag.core                       [INFO    ] meteoblue: provider ne

Recherche des images...
20 images trouvées


In [ ]:
# DOWNLOAD 
print("\nTéléchargement des ZIP...")

downloaded_files = []

for i, product in enumerate(search_results):
    try:
        print(f"[{i+1}/{len(search_results)}] {product.properties.get('title')}")

        path = dag.download(
            product,
            outputs_prefix=str(zip_dir),
            extract=False  
        )

        downloaded_files.append(path)

        print("OK téléchargé :", path)

    except Exception as e:
        print("Erreur téléchargement :", e)


Téléchargement des ZIP...
[1/20] S2B_MSIL2A_20230603T140709_N0510_R110_T21NZE_20240929T005954


0.00B [00:00, ?B/s]

2026-04-08 13:47:54,215 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(1adc6027-c8dc-4413-bed3-a62fdc39896a)/$value
2026-04-08 13:50:54,538 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 13:50:54,541 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(1adc6027-c8dc-4413-bed3-a62fdc39896a)/$value


OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230603T140709_N0510_R110_T21NZE_20240929T005954.zip
[2/20] S2B_MSIL2A_20230603T140709_N0510_R110_T21NZG_20240929T005954


0.00B [00:00, ?B/s]

...

0.00B [00:00, ?B/s]

2026-04-08 14:11:19,499 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(f2f8b81e-4b6e-4728-bec9-070384d99c0f)/$value
2026-04-08 14:18:31,925 eodag.download.base              [INFO    ] Extraction not activated. The product is available as is.
2026-04-08 14:18:31,928 eodag.product                    [INFO    ] Remote location of the product is still available through its 'remote_location' property: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(f2f8b81e-4b6e-4728-bec9-070384d99c0f)/$value


OK téléchargé : C:\Users\Kone\AppData\Local\Temp\S2B_MSIL2A_20230603T140709_N0510_R110_T22NBM_20240929T005954.zip
[9/20] S2A_MSIL2A_20230605T135711_N0510_R067_T22NDL_20240912T145232


0.00B [00:00, ?B/s]

2026-04-08 14:18:32,673 eodag.download.base              [INFO    ] Download url: https://catalogue.dataspace.copernicus.eu/odata/v1/Products(1dc73e58-c9de-46ca-8c09-09aa94c88c52)/$value


In [45]:
# EXTRACTION MANUELLE
print("\nExtraction des ZIP...")

for zip_path in downloaded_files:
    try:
        zip_path = Path(zip_path)

        print(f"Extraction : {zip_path.name}")

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print("OK extrait")

    except Exception as e:
        print("Erreur extraction :", e)

print("\nPipeline terminé proprement !")


Extraction des ZIP...
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T21NZE_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T21NZG_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T22NBK_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T21NZH_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T22NBL_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T21NZF_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T22NBN_20240929T005954.zip
OK extrait
Extraction : S2B_MSIL2A_20230603T140709_N0510_R110_T22NBM_20240929T005954.zip
OK extrait

Pipeline terminé proprement !


In [46]:
# DOSSIERS
raw_dir = Path("data/raw/Sentinel2/SAFE")
filtered_dir = Path("data/raw_filtered")
filtered_dir.mkdir(parents=True, exist_ok=True)

# seuil max de nuages (%)
CLOUD_THRESHOLD = 70

# classes nuages SCL
CLOUD_CLASSES = [3, 8, 9, 10]


# FONCTION CLOUD %
def compute_cloud_percentage(scl_path):
    with rasterio.open(scl_path) as src:
        scl = src.read(1)

    total_pixels = scl.size
    cloud_pixels = np.isin(scl, CLOUD_CLASSES).sum()
    cloud_percentage = (cloud_pixels / total_pixels) * 100

    return cloud_percentage


# PARCOURS DES IMAGES
print("\nAnalyse des nuages...")

kept = 0
skipped = 0

for safe in raw_dir.glob("*.SAFE"):

    try:
        scl_files = list(safe.glob("**/*SCL_20m.jp2"))
        if not scl_files:
            print(f"SCL non trouvé pour {safe.name}")
            continue

        scl_path = scl_files[0]
        cloud_pct = compute_cloud_percentage(scl_path)
        print(f"{safe.name} → {cloud_pct:.2f}% nuages")

        # FILTRAGE
        if cloud_pct <= CLOUD_THRESHOLD:
            # Déplacer vers le dossier filtré
            shutil.move(str(safe), filtered_dir)
            print(f"GARDÉ dans {filtered_dir}")
            kept += 1
        else:
            print("TROP nuageux, laissé dans raw")
            skipped += 1

    except Exception as e:
        print(f"Erreur avec {safe.name} :", e)

print("\nRésumé :")
print(f"Images gardées : {kept}")
print(f"Images laissées dans raw (trop nuageuses) : {skipped}")


Analyse des nuages...

Résumé :
Images gardées : 0
Images laissées dans raw (trop nuageuses) : 0
